# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
import joblib
from tqdm.notebook import tqdm
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


Создайте три пользовательских преобразователя, первые два из которых будут использоваться в рамках [конвейера](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. Класс `FeatureExtractor()`:
 - Извлекает из файла фрейм данных с `uid`, `labname`, `numTrials`, `timestamp` [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Извлекает "hour" из `timestamp`.
 - Извлекает "день недели" из "метки времени" (числа).
 - Удаляет столбец "метка времени".
 - Возвращает новый фрейм данных.


2. Класс `MyOneHotEncoder()`:
 - Берет фрейм данных из результата предыдущего преобразования и имя целевого столбца.
 - Идентифицирует все категориальные объекты и преобразует их с помощью OneHotEncoder(). Если целевой столбец также является категориальным, то преобразование не должно применяться к нему.
 - Удаляет исходные категориальные объекты.
 - Возвращает фрейм данных с объектами и серию с целевым столбцом.


3. Класс `TrainValidationTest()`:
 - Принимает значения `X` и `y`.
 - Возвращает `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).

In [2]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy['timestamp'] = pd.to_datetime(X_copy['timestamp'])
        X_copy['hour'] = X_copy['timestamp'].dt.hour
        X_copy['dayofweek'] = X_copy['timestamp'].dt.dayofweek
        X_copy = X_copy.drop(columns=['timestamp'])
        return X_copy

class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col):
        self.target_col = target_col
        self.cat_cols = []
        self.num_cols = []
        
        self.encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    def fit(self, X, y=None):
        if self.target_col in X.columns:
            features = X.drop(columns=[self.target_col])
        else:
            features = X
            
        self.cat_cols = [col for col in features.columns if features[col].dtype == 'object']
        self.num_cols = [col for col in features.columns if col not in self.cat_cols]
        
        self.encoder.fit(features[self.cat_cols])
        return self

    def transform(self, X):
        if self.target_col in X.columns:
            features = X.drop(columns=[self.target_col])
            target = X[self.target_col]
        else:
            features = X
            target = None
        
        encoded_cats = self.encoder.transform(features[self.cat_cols])
        
        feature_names = self.encoder.get_feature_names_out(self.cat_cols)
            
        encoded_df = pd.DataFrame(
            encoded_cats, 
            columns=feature_names,
            index=X.index
        )
        
        X_final = pd.concat([features[self.num_cols], encoded_df], axis=1)
        
        return X_final, target


In [3]:
class TrainValidationTest:
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def split(self):
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            self.X, self.y, 
            test_size=0.2, 
            random_state=21, 
            stratify=self.y
        )
        
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_full, y_train_full, 
            test_size=0.2, 
            random_state=21, 
            stratify=y_train_full
        )
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

Класс `ModelSelection()`

 - Получает список экземпляров `GridSearchCV` и dict, где ключами являются индексы из этого списка, а значениями - названия моделей, пример приведен ниже в обратном порядке (от высокоуровневого к низкоуровневому).:

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Метод `choose()` принимает значения `X_train`, `y_train`, `X_valid`, `y_valid` и возвращает имя наилучшего классификатора среди всех моделей в наборе проверки
 - Метод `best_results()` возвращает фрейм данных со столбцами `model`, `params`, `valid_score`, где строки являются лучшими моделями в каждом классе моделей.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - Когда вы выполняете итерацию по параметрам класса модели, выводите название этого класса и показывайте прогресс, используя "tqdm.notebook", в конце цикла выводите лучшую модель этого класса.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [4]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []
        self.best_overall_model_name = None
        self.best_overall_score = -1

    def choose(self, X_train, y_train, X_valid, y_valid):
        
        for idx, gs in enumerate(self.grids):
            model_name = self.grid_dict.get(idx, "Unknown")
            print(f"Estimator: {model_name}")
            
            with tqdm(total=1, desc=f"{model_name} progress") as pbar:
                gs.fit(X_train, y_train)
                pbar.update(1)
            
            best_params = gs.best_params_
            best_train_score = gs.best_score_
            
            best_model = gs.best_estimator_
            valid_pred = best_model.predict(X_valid)
            valid_score = accuracy_score(y_valid, valid_pred)
            
            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}\n")
            
            self.results.append({
                'model': model_name,
                'params': best_params,
                'valid_score': valid_score
            })
            
            if valid_score > self.best_overall_score:
                self.best_overall_score = valid_score
                self.best_overall_model_name = model_name

        print(f"Classifier with best validation set accuracy: {self.best_overall_model_name}")
        return self.best_overall_model_name

    def best_results(self):
        return pd.DataFrame(self.results)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

Класс `Finalize()`
 - Принимает значение оценки.
 - Метод `final_score()` принимает значения `X_train`, `y_train`, `X_test`, `y_test` и возвращает точность модели, как в примере ниже:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
- Метод `save_model()` определяет путь, сохраняет модель по этому пути и выводит сообщение о том, что модель была успешно сохранена.

In [5]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {acc}")
        return acc

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model successfully saved to {path}")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

1. Загрузите данные из файла (**** имя файла****).
2. Создайте конвейер предварительной обработки, состоящий из двух пользовательских преобразователей: `FeatureExtractor()` и `MyOneHotEncoder()`:
```
предварительная обработка = конвейер([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('день недели'))])
```
3. Используйте этот конвейер и его метод `fit_transform()` для исходного набора данных.
``
данные = предварительная обработка.fit_transform(df)
```
4. Получите `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test`, используя `TrainValidationTest()` и результат конвейера.
5. Создайте экземпляр `ModelSelection()`, используйте метод `choose()`, применяя его к желаемым моделям и параметрам, получите фрейм данных с наилучшими результатами.
6. создайте экземпляр `Finalize()` с вашей лучшей моделью, используйте метод `final_score()` и сохраните модель в формате: `name_of_the_model_{точность в тестовом наборе данных}.sav`.

Вот и все, поздравляю!

In [6]:
if __name__ == '__main__':
    df = pd.read_csv('../data/checker_submits.csv')
    
    preprocessing = Pipeline([
        ('feature_extractor', FeatureExtractor()),
        ('onehot_encoder', MyOneHotEncoder(target_col='dayofweek'))
    ])
    
    data = preprocessing.fit_transform(df)
    X, y = data


    
    splitter = TrainValidationTest(X, y)
    X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split()
    
    svm = SVC(random_state=21, probability=True)
    svm_params = {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto'],
        'class_weight': ['balanced', None]
    }
    gs_svm = GridSearchCV(svm, svm_params, scoring='accuracy', cv=3, n_jobs=-1)
    
    dt = DecisionTreeClassifier(random_state=21)
    dt_params = {
        'max_depth': [None, 10, 20],
        'criterion': ['gini', 'entropy'],
        'class_weight': ['balanced', None]
    }
    gs_dt = GridSearchCV(dt, dt_params, scoring='accuracy', cv=3, n_jobs=-1)
    
    rf = RandomForestClassifier(random_state=21)
    rf_params = {
        'n_estimators': [50, 100],
        'max_depth': [None, 10, 20],
        'criterion': ['gini', 'entropy']
    }
    gs_rf = GridSearchCV(rf, rf_params, scoring='accuracy', cv=3, n_jobs=-1)
    
    grids = [gs_svm, gs_dt, gs_rf]
    grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}
    
    selector = ModelSelection(grids, grid_dict)
    best_model_name = selector.choose(X_train, y_train, X_valid, y_valid)
    
    print("\nBest models results:")
    print(selector.best_results())
    
    best_grid_idx = [k for k, v in grid_dict.items() if v == best_model_name][0]
    best_estimator = grids[best_grid_idx].best_estimator_
    
    finalizer = Finalize(best_estimator)
    final_acc = finalizer.final_score(X_train, y_train, X_test, y_test)
    
    save_path = f"{best_model_name}_{final_acc:.5f}.sav"
    finalizer.save_model(save_path)

Estimator: SVM


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html